# Question Category Counts
Count all valid question categories across all datasets in `results/question_difficulty_complexity/`.

## 1. Import Required Libraries

In [30]:
import os
import json
import glob
from collections import Counter

import pandas as pd

In [31]:

# ── Discover all raw question_category values across the 5 dataset files ──────
from collections import defaultdict

DATA_DIR = "/Users/mominaahsan/Desktop/VisualTableBench/results/question_difficulty_complexity"

all_raw = defaultdict(set)

for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.json"))):
    name = os.path.splitext(os.path.basename(path))[0]
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        cat = item.get("question_category")
        if cat is not None:
            all_raw[name].add(cat)

# Per-file unique values
for name, cats in sorted(all_raw.items()):
    print(f"\n── {name.upper()} ({len(cats)} unique) ──")
    for c in sorted(cats):
        print(f"  {repr(c)}")

# Global unique values
global_unique = sorted({c for cats in all_raw.values() for c in cats})
print(f"\n{'='*60}")
print(f"ALL UNIQUE question_category VALUES  ({len(global_unique)} total)")
print(f"{'='*60}")
for c in global_unique:
    print(f"  {repr(c)}")



── FEVEROUS (19 unique) ──
  'Aggregation / Counting /'
  'Aggregation / Counting / Arithmetic'
  'Comparison &'
  'Comparison & Extremum'
  'Multi-'
  'Multi-Item Lookup'
  'Multi-hop'
  'Multi-hop Binary'
  'Multi-hop Binary Verification'
  'None of'
  'None of the'
  'None of the above'
  'None of the above —'
  'None of the above — Missing answer attribute in table'
  'Simple Lookup'
  'Single-'
  'Single-step'
  'Single-step Binary'
  'Single-step Binary Verification'

── HYBRIDQA (20 unique) ──
  'Aggregation'
  'Aggregation /'
  'Aggregation / Counting'
  'Aggregation / Counting /'
  'Aggregation / Counting / Arithmetic'
  'Comparison &'
  'Comparison & Extrem'
  'Comparison & Extremum'
  'Conditional Lookup'
  'Multi-'
  'Multi-Item'
  'Multi-Item Lookup'
  'None of'
  'None of the'
  'None of the above'
  'None of the above —'
  'None of the above — Missing answer attribute in table'
  'Simple Lookup'
  'Simple Lookup" ('
  'Single-'

── SQA (19 unique) ──
  'Aggregation'
  '

## 2. Define Category Mappings and Valid Categories

In [32]:

# All eight valid canonical categories.
VALID_CATEGORIES = {
    'Simple Lookup',
    'Conditional Lookup',
    'Multi-Item Lookup',
    'Aggregation / Counting / Arithmetic',
    'Comparison & Extremum',
    'Single-step Binary Verification',
    'Multi-hop Binary Verification',
    'None of the above',
}

print(f"Total valid categories: {len(VALID_CATEGORIES)}")
for cat in sorted(VALID_CATEGORIES):
    print(f"  • {cat}")

# Maps every observed raw string -> canonical name.
# Truncated / malformed variants are resolved to their full form.
# Truly ambiguous prefixes (e.g. 'Multi-' alone) are intentionally omitted.
CATEGORY_MAP = {
    # ── Simple Lookup ──────────────────────────────────────────────────────────
    'Simple Lookup':                                        'Simple Lookup',
    'Simple Lookup" (':                                     'Simple Lookup',
    'Simple Lookup) or "':                                  'Simple Lookup',

    # ── Conditional Lookup ─────────────────────────────────────────────────────
    'Conditional Lookup':                                   'Conditional Lookup',

    # ── Multi-Item Lookup ──────────────────────────────────────────────────────
    'Multi-':                                               'Multi-Item Lookup',
    'Multi-Item':                                           'Multi-Item Lookup',
    'Multi-Item Lookup':                                    'Multi-Item Lookup',

    # ── Aggregation / Counting / Arithmetic ────────────────────────────────────
    'Aggregation':                                          'Aggregation / Counting / Arithmetic',
    'Aggregation /':                                        'Aggregation / Counting / Arithmetic',
    'Aggregation / Counting':                               'Aggregation / Counting / Arithmetic',
    'Aggregation / Counting /':                             'Aggregation / Counting / Arithmetic',
    'Aggregation / Counting / Arithmetic':                  'Aggregation / Counting / Arithmetic',

    # ── Comparison & Extremum ──────────────────────────────────────────────────
    'Comparison &':                                         'Comparison & Extremum',
    'Comparison & Extrem':                                  'Comparison & Extremum',
    'Comparison & Extremum':                                'Comparison & Extremum',

    # ── Single-step Binary Verification ───────────────────────────────────────
    'Single':                                               'Single-step Binary Verification',
    'Single-':                                              'Single-step Binary Verification',
    'Single-step':                                          'Single-step Binary Verification',
    'Single-step Binary':                                   'Single-step Binary Verification',
    'Single-step Binary Verification':                      'Single-step Binary Verification',
    'Single-step binary':                                   'Single-step Binary Verification',

    # ── Multi-hop Binary Verification ─────────────────────────────────────────
    'Multi-hop':                                            'Multi-hop Binary Verification',
    'Multi-hop Binary':                                     'Multi-hop Binary Verification',
    'Multi-hop Binary Verification':                        'Multi-hop Binary Verification',

    # ── None of the above ─────────────────────────────────────────────────────
    'None of':                                              'None of the above',
    'None of the':                                          'None of the above',
    'None of the above':                                    'None of the above',
    'None of the above —':                                  'None of the above',
    'None of the above — Missing answer attribute in table':'None of the above',
    'None of the above — Not table-required':               'None of the above',
}

print(f"\nCategory map defined with {len(CATEGORY_MAP)} entries.")
print("Note: 'Multi-' (ambiguous prefix) is intentionally left unmapped.")


Total valid categories: 8
  • Aggregation / Counting / Arithmetic
  • Comparison & Extremum
  • Conditional Lookup
  • Multi-Item Lookup
  • Multi-hop Binary Verification
  • None of the above
  • Simple Lookup
  • Single-step Binary Verification

Category map defined with 30 entries.
Note: 'Multi-' (ambiguous prefix) is intentionally left unmapped.


## 3. Load and Parse JSON Files

In [33]:
DATA_DIR = "/Users/mominaahsan/Desktop/VisualTableBench/results/question_difficulty_complexity"

json_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.json")))
print(f"Found {len(json_files)} JSON file(s):")
for p in json_files:
    print(" ", os.path.basename(p))

# Collect raw category strings per dataset
per_dataset_raw = {}
for path in json_files:
    dataset_name = os.path.splitext(os.path.basename(path))[0]
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    per_dataset_raw[dataset_name] = [item.get("question_category") for item in data]

total_entries = sum(len(v) for v in per_dataset_raw.values())
print(f"\nTotal entries loaded: {total_entries}")

Found 5 JSON file(s):
  feverous.json
  hybridqa.json
  sqa.json
  tabfact.json
  wikitq.json

Total entries loaded: 6097


## 4. Count and Filter Valid Categories

In [34]:
# --- Overall totals ---
overall_counter = Counter()

# --- Per-dataset breakdown ---
per_dataset_counts = {}

for dataset_name, raw_categories in per_dataset_raw.items():
    ds_counter = Counter()
    for raw in raw_categories:
        if raw is None:
            continue
        canonical = CATEGORY_MAP.get(raw)          # None if not in map
        if canonical is None:
            continue                                # unknown / excluded category
        if canonical not in VALID_CATEGORIES:
            continue                                # e.g. Arithmetic Calculation
        ds_counter[canonical] += 1
        overall_counter[canonical] += 1
    per_dataset_counts[dataset_name] = ds_counter

print("Counting complete.")

Counting complete.


## 5. Display Results

In [35]:
# ── Per-dataset breakdown ──────────────────────────────────────────────────────
print("=" * 70)
print("PER-DATASET BREAKDOWN")
print("=" * 70)

for dataset_name, counter in per_dataset_counts.items():
    df_ds = (
        pd.DataFrame(counter.items(), columns=["Category", "Count"])
        .sort_values("Count", ascending=False)
        .reset_index(drop=True)
    )
    print(f"\n── {dataset_name.upper()} (total valid: {df_ds['Count'].sum()}) ──")
    display(df_ds)

# ── Overall totals ─────────────────────────────────────────────────────────────
print("=" * 70)
print("OVERALL TOTALS (all datasets combined)")
print("=" * 70)

df_overall = (
    pd.DataFrame(overall_counter.items(), columns=["Category", "Count"])
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)
display(df_overall)

grand_total = df_overall["Count"].sum()
print(f"\nGrand total valid questions: {grand_total}")

PER-DATASET BREAKDOWN

── FEVEROUS (total valid: 794) ──


,Category,Count
0,Single-step Binary Verification,305
1,Multi-hop Binary Verification,294
2,None of the above,114
3,Multi-Item Lookup,69
4,Comparison & Extremum,7
5,Aggregation / Counting / Arithmetic,4
6,Simple Lookup,1



── HYBRIDQA (total valid: 1608) ──


,Category,Count
0,Conditional Lookup,746
1,None of the above,611
2,Simple Lookup,88
3,Multi-Item Lookup,67
4,Comparison & Extremum,48
5,Aggregation / Counting / Arithmetic,47
6,Single-step Binary Verification,1



── SQA (total valid: 1000) ──


...


── WIKITQ (total valid: 1000) ──


,Category,Count
0,Aggregation / Counting / Arithmetic,408
1,Comparison & Extremum,234
2,Simple Lookup,144
3,Conditional Lookup,139
4,Multi-Item Lookup,35
5,None of the above,22
6,Single-step Binary Verification,15
7,Multi-hop Binary Verification,3


OVERALL TOTALS (all datasets combined)


,Category,Count
0,Single-step Binary Verification,1483
1,Conditional Lookup,1005
2,None of the above,895
3,Multi-Item Lookup,803
4,Multi-hop Binary Verification,599
5,Aggregation / Counting / Arithmetic,509
6,Comparison & Extremum,459
7,Simple Lookup,343



Grand total valid questions: 6096


In [ ]:
## 6. Category Counts by Difficulty (Easy vs Hard)

In [36]:
# Re-read files to collect (category, difficulty) pairs
difficulty_counter = {"Easy": Counter(), "Hard": Counter()}

for path in json_files:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        raw_cat  = item.get("question_category")
        raw_diff = item.get("question_difficulty")
        if raw_cat is None or raw_diff not in ("Easy", "Hard"):
            continue
        canonical = CATEGORY_MAP.get(raw_cat)
        if canonical is None or canonical not in VALID_CATEGORIES:
            continue
        difficulty_counter[raw_diff][canonical] += 1

# ── Build a combined comparison DataFrame ─────────────────────────────────────
all_cats = sorted(VALID_CATEGORIES)

df_diff = pd.DataFrame({
    "Category":  all_cats,
    "Easy":      [difficulty_counter["Easy"].get(c, 0) for c in all_cats],
    "Hard":      [difficulty_counter["Hard"].get(c, 0) for c in all_cats],
})
df_diff["Total"] = df_diff["Easy"] + df_diff["Hard"]
df_diff = df_diff.sort_values("Total", ascending=False).reset_index(drop=True)

# ── Easy breakdown ─────────────────────────────────────────────────────────────
print("=" * 70)
print("DIFFICULTY: EASY")
print("=" * 70)
df_easy = (
    pd.DataFrame(difficulty_counter["Easy"].items(), columns=["Category", "Count"])
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)
display(df_easy)
print(f"Total Easy: {df_easy['Count'].sum()}")

# ── Hard breakdown ─────────────────────────────────────────────────────────────
print("=" * 70)
print("DIFFICULTY: HARD")
print("=" * 70)
df_hard = (
    pd.DataFrame(difficulty_counter["Hard"].items(), columns=["Category", "Count"])
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)
display(df_hard)
print(f"Total Hard: {df_hard['Count'].sum()}")

# ── Side-by-side summary ───────────────────────────────────────────────────────
print("=" * 70)
print("EASY vs HARD — SIDE-BY-SIDE SUMMARY")
print("=" * 70)
display(df_diff)
print(f"\nEasy total: {df_diff['Easy'].sum()}  |  Hard total: {df_diff['Hard'].sum()}")

DIFFICULTY: EASY


,Category,Count
0,Single-step Binary Verification,1359
1,Conditional Lookup,559
2,Multi-hop Binary Verification,517
3,Multi-Item Lookup,445
4,Aggregation / Counting / Arithmetic,362
5,Comparison & Extremum,306
6,Simple Lookup,251
7,None of the above,174


Total Easy: 3973
DIFFICULTY: HARD


,Category,Count
0,None of the above,721
1,Conditional Lookup,446
2,Multi-Item Lookup,358
3,Comparison & Extremum,153
4,Aggregation / Counting / Arithmetic,147
5,Single-step Binary Verification,124
6,Simple Lookup,92
7,Multi-hop Binary Verification,82


Total Hard: 2123
EASY vs HARD — SIDE-BY-SIDE SUMMARY


,Category,Easy,Hard,Total
0,Single-step Binary Verification,1359,124,1483
1,Conditional Lookup,559,446,1005
2,None of the above,174,721,895
3,Multi-Item Lookup,445,358,803
4,Multi-hop Binary Verification,517,82,599
5,Aggregation / Counting / Arithmetic,362,147,509
6,Comparison & Extremum,306,153,459
7,Simple Lookup,251,92,343



Easy total: 3973  |  Hard total: 2123


# BUILD FINAL DATASET

In [40]:
import random
from collections import defaultdict
from pathlib import Path

SOURCE_DIR  = Path("/Users/mominaahsan/Desktop/VisualTableBench/results/question_difficulty_complexity")
OUTPUT_FILE = Path("/Users/mominaahsan/Desktop/VisualTableBench/data/1-raw/data_new.json")

# Categories to sample from (excludes 'None of the above')
SAMPLE_CATEGORIES = [
    'Simple Lookup',
    'Conditional Lookup',
    'Multi-Item Lookup',
    'Aggregation / Counting / Arithmetic',
    'Comparison & Extremum',
    'Single-step Binary Verification',
    'Multi-hop Binary Verification',
]

# Build pool[difficulty][category] = list of items
pool = defaultdict(lambda: defaultdict(list))

for path in sorted(SOURCE_DIR.glob("*.json")):
    with open(path, encoding="utf-8") as f:
        items = json.load(f)
    for item in items:
        raw_cat  = item.get("question_category", "")
        raw_diff = item.get("question_difficulty", "")
        if raw_diff not in ("Easy", "Hard"):
            continue
        canonical = CATEGORY_MAP.get(raw_cat)
        if canonical is None or canonical not in SAMPLE_CATEGORIES:
            continue
        item = dict(item)                          # shallow copy — don't mutate source
        item["question_category"] = canonical      # store canonical name
        item["dataset"] = path.stem
        pool[raw_diff][canonical].append(item)

# ── Pool sizes ────────────────────────────────────────────────────────────────
print(f"{'Category':<45}  {'Easy':>6}  {'Hard':>6}  {'Total':>6}")
print("-" * 65)
for cat in SAMPLE_CATEGORIES:
    e = len(pool["Easy"][cat])
    h = len(pool["Hard"][cat])
    print(f"{cat:<45}  {e:>6}  {h:>6}  {e+h:>6}")


Category                                         Easy    Hard   Total
-----------------------------------------------------------------
Simple Lookup                                     251      92     343
Conditional Lookup                                559     446    1005
Multi-Item Lookup                                 445     358     803
Aggregation / Counting / Arithmetic               362     147     509
Comparison & Extremum                             306     153     459
Single-step Binary Verification                  1359     124    1483
Multi-hop Binary Verification                     517      82     599


## 9. Phase 1 — Sample Unique image_ids
For each category × difficulty, build the available pool by keeping only one sample per `image_id` (no cross-category reuse either). Sample up to 50. EASY runs first; its used `image_id`s carry over to HARD.

In [41]:
TARGET = 50          # samples per category per difficulty

def phase1_sample(cat_pool, used_image_ids, label):
    """
    Pick up to TARGET samples with unique image_ids.
    - No image_id must appear twice within this category.
    - No image_id already in used_image_ids (cross-category / cross-difficulty).
    Returns (selected_list, new_image_ids_set).
    """
    available = []
    seen_here = set()
    for item in cat_pool:
        iid = item["image_id"]
        if iid not in used_image_ids and iid not in seen_here:
            available.append(item)
            seen_here.add(iid)
    take = min(TARGET, len(available))
    selected = random.sample(available, take)
    new_ids = {s["image_id"] for s in selected}
    print(f"  [{label}] {selected[0]['question_category'] if selected else '?':<45}"
          f"  sampled {take:3}/{TARGET}  (unique pool: {len(available)})")
    return selected, new_ids

random.seed(42)

used_image_ids = set()          # accumulates across ALL categories + difficulties
easy_bucket = {}                # cat -> list of selected items
hard_bucket = {}

print("=" * 72)
print("PHASE 1 — EASY")
print("=" * 72)
for cat in SAMPLE_CATEGORIES:
    sel, new_ids = phase1_sample(pool["Easy"][cat], used_image_ids, "EASY")
    easy_bucket[cat] = sel
    used_image_ids.update(new_ids)

print(f"\n  Total Easy (phase 1): {sum(len(v) for v in easy_bucket.values())}")

print("\n" + "=" * 72)
print("PHASE 1 — HARD")
print("=" * 72)
for cat in SAMPLE_CATEGORIES:
    sel, new_ids = phase1_sample(pool["Hard"][cat], used_image_ids, "HARD")
    hard_bucket[cat] = sel
    used_image_ids.update(new_ids)

print(f"\n  Total Hard (phase 1): {sum(len(v) for v in hard_bucket.values())}")


PHASE 1 — EASY
  [EASY] Simple Lookup                                  sampled  50/50  (unique pool: 209)
  [EASY] Conditional Lookup                             sampled  50/50  (unique pool: 506)
  [EASY] Multi-Item Lookup                              sampled  50/50  (unique pool: 283)
  [EASY] Aggregation / Counting / Arithmetic            sampled  50/50  (unique pool: 268)
  [EASY] Comparison & Extremum                          sampled  50/50  (unique pool: 231)
  [EASY] Single-step Binary Verification                sampled  50/50  (unique pool: 1302)
  [EASY] Multi-hop Binary Verification                  sampled  50/50  (unique pool: 460)

  Total Easy (phase 1): 350

PHASE 1 — HARD
  [HARD] Simple Lookup                                  sampled  50/50  (unique pool: 74)
  [HARD] Conditional Lookup                             sampled  50/50  (unique pool: 430)
  [HARD] Multi-Item Lookup                              sampled  50/50  (unique pool: 172)
  [HARD] Aggregation / Countin

## 10. Phase 2 — Upsize Short Categories
For any category that fell below 50 after Phase 1, fill the remaining slots by freely sampling from the remainder of the pool (duplicate `image_id`s are allowed here).

In [42]:
def phase2_upsize(cat, cat_pool, current, label):
    """
    Fill (TARGET - len(current)) slots from cat_pool items not already selected.
    Duplicate image_ids are allowed. Uses object identity to avoid exact duplicates.
    Returns list of additional items.
    """
    needed = TARGET - len(current)
    if needed <= 0:
        return []
    selected_ids = {id(s) for s in current}
    remainder = [s for s in cat_pool if id(s) not in selected_ids]
    take = min(needed, len(remainder))
    extra = random.sample(remainder, take)
    print(f"  [{label}] {cat:<45}  +{take:2}  (now {len(current)+take}/{TARGET},  remainder pool: {len(remainder)})")
    return extra

upsized_any = False

print("=" * 72)
print("PHASE 2 — UPSIZING")
print("=" * 72)
for cat in SAMPLE_CATEGORIES:
    if len(easy_bucket[cat]) < TARGET:
        upsized_any = True
        extra = phase2_upsize(cat, pool["Easy"][cat], easy_bucket[cat], "EASY")
        easy_bucket[cat].extend(extra)

for cat in SAMPLE_CATEGORIES:
    if len(hard_bucket[cat]) < TARGET:
        upsized_any = True
        extra = phase2_upsize(cat, pool["Hard"][cat], hard_bucket[cat], "HARD")
        hard_bucket[cat].extend(extra)

if not upsized_any:
    print("  All categories already at target — no upsizing needed.")

print(f"\n  Total Easy after upsize : {sum(len(v) for v in easy_bucket.values())}")
print(f"  Total Hard after upsize : {sum(len(v) for v in hard_bucket.values())}")


PHASE 2 — UPSIZING
  All categories already at target — no upsizing needed.

  Total Easy after upsize : 350
  Total Hard after upsize : 350


## 11. Save Final Dataset
Combine Easy + Hard, deduplicate on `(table, query)`, assign sequential `new_id`, and write to `data/1-raw/data.json`.

In [43]:
# ── Combine ───────────────────────────────────────────────────────────────────
easy_all = [s for cat in SAMPLE_CATEGORIES for s in easy_bucket[cat]]
hard_all = [s for cat in SAMPLE_CATEGORIES for s in hard_bucket[cat]]
combined = easy_all + hard_all

# ── Deduplicate on (table JSON, query) ────────────────────────────────────────
seen_keys = {}
deduped = []
for item in combined:
    tkey = json.dumps(item.get("table", {}), sort_keys=True, ensure_ascii=False)
    qkey = item.get("query", "")
    k = (tkey, qkey)
    if k not in seen_keys:
        seen_keys[k] = True
        deduped.append(item)

removed = len(combined) - len(deduped)
if removed:
    print(f"⚠  Removed {removed} duplicate (table, query) pair(s).")
else:
    print(f"✓  No duplicates — all {len(deduped)} samples are unique.")

# ── Assign new_id ─────────────────────────────────────────────────────────────
final_dataset = []
for idx, item in enumerate(deduped):
    new_item = {"new_id": idx}
    new_item.update(item)
    final_dataset.append(new_item)

# ── Save ──────────────────────────────────────────────────────────────────────
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"✓  Saved {len(final_dataset)} samples → {OUTPUT_FILE}")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print(f"{'Category':<45}  {'Easy':>5}  {'Hard':>5}  {'Total':>6}")
print("-" * 72)
for cat in SAMPLE_CATEGORIES:
    e = len(easy_bucket[cat])
    h = len(hard_bucket[cat])
    flag = " ⚠" if e < TARGET or h < TARGET else ""
    print(f"{cat:<45}  {e:>5}  {h:>5}  {e+h:>6}{flag}")
print("-" * 72)
total_e = sum(len(easy_bucket[c]) for c in SAMPLE_CATEGORIES)
total_h = sum(len(hard_bucket[c]) for c in SAMPLE_CATEGORIES)
print(f"{'TOTAL':<45}  {total_e:>5}  {total_h:>5}  {total_e+total_h:>6}")
print(f"\n  Unique image_ids used (phase 1): {len(used_image_ids)}")
print(f"  new_id range: 0 → {len(final_dataset)-1}")


✓  No duplicates — all 700 samples are unique.
✓  Saved 700 samples → /Users/mominaahsan/Desktop/VisualTableBench/data/1-raw/data_new.json

Category                                        Easy   Hard   Total
------------------------------------------------------------------------
Simple Lookup                                     50     50     100
Conditional Lookup                                50     50     100
Multi-Item Lookup                                 50     50     100
Aggregation / Counting / Arithmetic               50     50     100
Comparison & Extremum                             50     50     100
Single-step Binary Verification                   50     50     100
Multi-hop Binary Verification                     50     50     100
------------------------------------------------------------------------
TOTAL                                            350    350     700

  Unique image_ids used (phase 1): 700
  new_id range: 0 → 699


# FINAL COUNTS

In [47]:

# Count total number of samples per category per difficulty level (easy vs hard) in data.json

DATA_JSON_PATH = "/Users/mominaahsan/Desktop/VisualTableBench/data/1-raw/data.json"

with open(DATA_JSON_PATH, encoding="utf-8") as f:
    data_json = json.load(f)

diff_cat_counter = {"Easy": Counter(), "Hard": Counter()}

for item in data_json:
    raw_cat  = item.get("question_category", "")
    raw_diff = item.get("question_difficulty", "")
    if raw_diff not in ("Easy", "Hard"):
        continue
    canonical = CATEGORY_MAP.get(raw_cat, raw_cat)   # fall back to raw if unmapped
    diff_cat_counter[raw_diff][canonical] += 1

all_cats = sorted({c for ctr in diff_cat_counter.values() for c in ctr})

df_counts = pd.DataFrame({
    "Category": all_cats,
    "Easy":     [diff_cat_counter["Easy"].get(c, 0) for c in all_cats],
    "Hard":     [diff_cat_counter["Hard"].get(c, 0) for c in all_cats],
})
df_counts["Total"] = df_counts["Easy"] + df_counts["Hard"]
df_counts = df_counts.sort_values("Total", ascending=False).reset_index(drop=True)

print(f"{'='*70}")
print(f"SAMPLES PER CATEGORY × DIFFICULTY  —  data.json  ({len(data_json)} total)")
print(f"{'='*70}")
display(df_counts)
print(f"\nEasy total : {df_counts['Easy'].sum()}")
print(f"Hard total : {df_counts['Hard'].sum()}")
print(f"Grand total: {df_counts['Total'].sum()}")
#print unique image_id counts
unique_image_ids = set(item.get("image_id") for item in data_json if "image_id" in item)
print(f"Unique image_ids in data.json: {len(unique_image_ids)}")


SAMPLES PER CATEGORY × DIFFICULTY  —  data.json  (700 total)


,Category,Easy,Hard,Total
0,Aggregation / Counting / Arithmetic,50,50,100
1,Comparison & Extremum,50,50,100
2,Conditional Lookup,50,50,100
3,Multi-Item Lookup,50,50,100
4,Multi-hop Binary Verification,50,50,100
5,Simple Lookup,50,50,100
6,Single-step Binary Verification,50,50,100



Easy total : 350
Hard total : 350
Grand total: 700
Unique image_ids in data.json: 629
